# 01 · FONDMENS + FONDREGISTRO: NAV and metadata anomaly profiling

**Objective:** measure how many anomalies Silver contains in NAV and metadata, and where.

**Scope.** Silver covers the two mass sources, FONDMENS + FONDREGISTRO, materialized as `fund_nav_daily` + `fund_metadata`.


In [1]:
import duckdb

ROOT = "../.."  # repo root, relative to this notebook's folder (notebooks/exploration)
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW nav AS
               SELECT * FROM read_parquet('{ROOT}/data/silver/fund_nav_daily/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW meta AS
               SELECT * FROM read_parquet('{ROOT}/data/silver/fund_metadata/**/*.parquet', hive_partitioning=true)""")

# Reproducibility header: the exact slice of Silver this run measured
print("Reproducibility header: slice of Silver measured in this run")
duckdb.sql("""
    SELECT (SELECT count(*) FROM nav)                          AS nav_rows,
           (SELECT count(*) FROM meta)                         AS meta_rows,
           (SELECT min(FechaDatos) || '..' || max(FechaDatos) FROM nav) AS nav_periods,
           (SELECT count(DISTINCT FechaDatos) FROM nav)        AS n_periods,
           current_date                                        AS executed_on
""").df()

Reproducibility header: slice of Silver measured in this run


,nav_rows,meta_rows,nav_periods,n_periods,executed_on
0,3806718,125807,202301..202605,41,2026-10-22


## Data preview
**Question:** what does one row actually look like? Three sample rows from each table, so the column names used throughout the notebook have a concrete shape.

In [2]:
print("fund_nav_daily (3 rows)")
display(duckdb.sql("SELECT * FROM nav LIMIT 3").df())
print("fund_metadata (3 rows)")
display(duckdb.sql("SELECT * FROM meta LIMIT 3").df())

fund_nav_daily (3 rows)


,Tipo,NumeroRegistro,NumeroCompartimento,NumeroClase,ISIN,value_date,VL,Participes,Patrimonio,FechaDatos
0,FI,9,0,1,ES0138841038,2023-01-01,27.8142,0,58952984.33,202301
1,FI,9,0,1,ES0138841038,2023-01-02,27.8364,1647,58999976.55,202301
2,FI,9,0,1,ES0138841038,2023-01-03,27.8496,1646,59074086.62,202301


fund_metadata (3 rows)


,Tipo,NumeroRegistro,Denominacion,ETF,NumeroCompartimento,DenominacionCompartimento,NumeroClase,DenominacionClase,ISIN,DenominacionGestora,NumeroRegistroGestora,TipoGestora,DenominacionGrupoGestora,DenominacionDepositario,NumeroRegistroDepositario,DenominacionGrupoDepositario,FechaDatos
0,FI,9,"FONMARCH, FI",False,0,COMPARTIMENTO 0,1,CLASE A,ES0138841038,"MARCH ASSET MANAGEMENT, S.G.I.I.C., S.A.U.",190,SGIIC,BANCA MARCH,"BANCA MARCH, S.A.",10,BANCA MARCH,202301
1,FI,9,"FONMARCH, FI",False,0,COMPARTIMENTO 0,2,CLASE C,ES0138841004,"MARCH ASSET MANAGEMENT, S.G.I.I.C., S.A.U.",190,SGIIC,BANCA MARCH,"BANCA MARCH, S.A.",10,BANCA MARCH,202301
2,FI,9,"FONMARCH, FI",False,0,COMPARTIMENTO 0,3,CLASE S,ES0138841012,"MARCH ASSET MANAGEMENT, S.G.I.I.C., S.A.U.",190,SGIIC,BANCA MARCH,"BANCA MARCH, S.A.",10,BANCA MARCH,202301


## 1. Scale
**Question:** how big is the dataset we are profiling?

In [3]:
print("Full history scale")
display(
    duckdb.sql("""
    SELECT count(*) AS n_rows,
           count(DISTINCT FechaDatos) AS periods,
           count(DISTINCT ISIN) AS isins,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) AS class_keys
    FROM nav
""").df()
)

# Latest snapshot scale: last published day of the last period
print("Latest snapshot scale (last published day)")
duckdb.sql("""
    SELECT max(FechaDatos) AS latest_period,
           count(DISTINCT (Tipo, NumeroRegistro)) AS funds_with_positive_aum,
           round(sum(Patrimonio) / 1e9, 1) AS total_aum_bn
    FROM nav
    WHERE FechaDatos = (SELECT max(FechaDatos) FROM nav)
      AND value_date = (SELECT max(value_date) FROM nav WHERE FechaDatos = (SELECT max(FechaDatos) FROM nav))
      AND Patrimonio > 0 AND VL > 0
""").df()

Full history scale


,n_rows,periods,isins,class_keys
0,3806718,41,3769,3857


Latest snapshot scale (last published day)


,latest_period,funds_with_positive_aum,total_aum_bn
0,202605,1430,485.5


- **Note:** the ISIN count is slightly lower than the class-key count, which indicates that some ISINs are shared across keys. [§7](#7-referential-integrity-and-isin-quality) confirms this and explains why the source allows it.

## 2. Anomaly flags (raw counts)
**Question:** how many rows violate each rule?

These are **non-exclusive** counters, the exact predicates the gate uses. One row can violate several rules at once (for example, `VL = 0` together with `Patrimonio < 0` counts in both flags). No priority, no rows hidden.

In [4]:
# Flags and duplicate keys: the exact predicates the gate uses
flags = duckdb.sql("""
    SELECT count(*) FILTER (WHERE VL IS NULL OR Patrimonio IS NULL OR Participes IS NULL) AS any_null,
           count(*) FILTER (WHERE VL < 0)         AS negative_vl,
           count(*) FILTER (WHERE Patrimonio < 0) AS negative_aum,
           count(*) FILTER (WHERE Participes < 0) AS negative_holders,
           count(*) FILTER (WHERE VL = 0)         AS zero_vl,
           count(*) FILTER (WHERE Patrimonio = 0) AS zero_aum,
           count(*) FILTER (WHERE Participes = 0) AS zero_holders
    FROM nav
""").df()
dup_key_groups = duckdb.sql("""
    SELECT count(*) FROM (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase, value_date
        FROM nav GROUP BY ALL HAVING count(*) > 1
    )
""").fetchone()[0]
print(f"duplicate key groups (expected 0): {dup_key_groups}")
print("Raw anomaly flag counts (non-exclusive)")
display(flags)

duplicate key groups (expected 0): 0
Raw anomaly flag counts (non-exclusive)


,any_null,negative_vl,negative_aum,negative_holders,zero_vl,zero_aum,zero_holders
0,0,5,522,10,202963,200512,216205


- **Note:** the flags split into two families.
  - Negative values are rare: at most 522 rows against 3.8 million, negligible in volume.
  - Zeros are the material anomaly, close to 5% of the dataset.
- **Hypothesis:** the zeros combine padding with dormant classes that never publish a positive NAV. Their share is stable every month ([§5](#5-stability-across-periods)), which points to a property of the source rather than to corruption.
- **Gate:** negatives are FAIL, zeros are WARN. All pre-existing.

## 3. What *can* we check? (sign coherence)

Each fund-day carries three signed numbers: `VL` (NAV per unit), `Participes` (number of investors) and `Patrimonio` (AUM). Not every combination of signs is possible in a coherent fund:

| Combination | Plain reading | Why it is incoherent |
|---|---|---|
| AUM positive, NAV zero or negative | the fund holds money but each unit is worthless | a solvent fund cannot have both |
| AUM positive, investors zero or negative | the fund holds money but nobody owns it | assets cannot exist without owners |
| NAV positive, investors positive, AUM zero or negative | investors hold valuable units but the pool is empty | totals contradict the parts |

The tables below count how often each incoherent combination occurs, on how many distinct funds, and on which days.

One caveat: do not check `Patrimonio = VL × Participes`. `Participes` counts investors, not units outstanding, so the multiplication is never valid for this source.

In [5]:
# Every fully populated row lands in exactly one of the 8 sign cells
sign_matrix = duckdb.sql("""
    SELECT (VL > 0)         AS vl_positive,
           (Participes > 0) AS holders_positive,
           (Patrimonio > 0) AS aum_positive,
           count(*) AS n_rows,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) AS n_funds
    FROM nav
    WHERE VL IS NOT NULL AND Participes IS NOT NULL AND Patrimonio IS NOT NULL
    GROUP BY ALL
    ORDER BY 1, 2, 3
""").df()
print("Sign matrix: every fully populated row in one of 8 cells")
display(sign_matrix)

# Contradictory combinations only: counts and fund unions (promoted to the gate as WARN)
print("Contradictory combinations: rows and distinct funds")
display(
    duckdb.sql("""
    SELECT count(*) FILTER (WHERE Patrimonio > 0 AND VL <= 0)         AS aum_no_nav_rows,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE Patrimonio > 0 AND VL <= 0)         AS aum_no_nav_funds,
           count(*) FILTER (WHERE Patrimonio > 0 AND Participes <= 0) AS aum_no_holders_rows,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE Patrimonio > 0 AND Participes <= 0) AS aum_no_holders_funds,
           count(*) FILTER (WHERE VL > 0 AND Participes > 0 AND Patrimonio <= 0) AS nav_no_aum_rows,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE VL > 0 AND Participes > 0 AND Patrimonio <= 0) AS nav_no_aum_funds
    FROM nav
""").df()
)

# Which days carry the most contradictions?
print("Busiest contradiction days")
duckdb.sql("""
    SELECT value_date, count(*) AS contradiction_rows
    FROM nav
    WHERE (Patrimonio > 0 AND VL <= 0)
       OR (Patrimonio > 0 AND Participes <= 0)
       OR (VL > 0 AND Participes > 0 AND Patrimonio <= 0)
    GROUP BY value_date
    ORDER BY contradiction_rows DESC
    LIMIT 5
""").df()

Sign matrix: every fully populated row in one of 8 cells


,vl_positive,holders_positive,aum_positive,n_rows,n_funds
0,False,False,False,200495,1070
1,False,False,True,1293,80
2,False,True,False,393,36
3,False,True,True,787,593
4,True,False,False,135,123
5,True,False,True,14292,665
6,True,True,False,11,10
7,True,True,True,3589312,3713


Contradictory combinations: rows and distinct funds


,aum_no_nav_rows,aum_no_nav_funds,aum_no_holders_rows,aum_no_holders_funds,nav_no_aum_rows,nav_no_aum_funds
0,2080,656,15585,706,11,10


Busiest contradiction days


,value_date,contradiction_rows
0,2025-12-25,149
1,2025-05-01,146
2,2023-10-28,129
3,2023-10-29,129
4,2023-08-27,118


In [6]:
# Derived quantity with no published counterpart: implied units = AUM / NAV (distribution, not a rule)
print("Implied units outstanding (AUM / NAV): distribution, not a rule")
duckdb.sql("""
    SELECT count(*) FILTER (WHERE VL > 0 AND Patrimonio > 0) AS both_positive_rows,
           round(min(Patrimonio / VL) FILTER (WHERE VL > 0 AND Patrimonio > 0), 2)               AS implied_units_min,
           round(quantile_cont(Patrimonio / VL, 0.5) FILTER (WHERE VL > 0 AND Patrimonio > 0), 2)  AS implied_units_p50,
           round(quantile_cont(Patrimonio / VL, 0.99) FILTER (WHERE VL > 0 AND Patrimonio > 0), 2) AS implied_units_p99
    FROM nav
""").df()

Implied units outstanding (AUM / NAV): distribution, not a rule


,both_positive_rows,implied_units_min,implied_units_p50,implied_units_p99
0,3603604,0.0,752969.81,1.841602e+08


- **Note:** the three contradictory combinations differ wildly in volume: the rarest has 11 rows, while the other two reach the tens of thousands. The third table shows that the busiest days each carry between 118 and 149 contradictory rows.
- **Hypothesis:** the contradictions are not scattered; they cluster on specific days across many periods, which suggests a recurring reporting pattern in the source rather than a single broken month. The dominant pattern, positive AUM and NAV with zero investors, suggests the investors field is updated on a different cadence than NAV and AUM, or stops being refreshed for classes that keep trading. The implied-units distribution stays plausible throughout, so AUM and NAV remain mutually credible even when holders look inconsistent.
- **Gate:** all three combinations were promoted as WARN.

## 4. Distributions and sentinels

Two questions: are the value ranges plausible, and which repeated non-zero values are sentinels?

A sentinel value is one that reveals a system behaviour rather than market data. The diagnostic is the number of distinct funds sharing the value: a value repeated across many funds is a systemic property of the source (a vehicle type, a reporting convention); a value repeated on a single fund is idiosyncratic (a suspension, a particular fund event).

The first table shows percentiles of the positive range per measure and the distance to the Silver decimal ceilings. The second table shows, per measure, the five most frequent non-zero values with their occurrence and fund counts, the raw material for the sentinel reading.

In [7]:
# Percentiles of the positive range, per measure
print("Percentiles of the positive range, per measure")
display(
    duckdb.sql("""
    SELECT 'VL' AS measure, min(v) AS min_positive,
           quantile_cont(v, 0.01) AS p01, quantile_cont(v, 0.5) AS p50,
           quantile_cont(v, 0.99) AS p99, quantile_cont(v, 0.999) AS p999,
           max(v) AS max_value
    FROM (SELECT VL AS v FROM nav WHERE VL > 0)
    UNION ALL
    SELECT 'Patrimonio', min(p),
           quantile_cont(p, 0.01), quantile_cont(p, 0.5),
           quantile_cont(p, 0.99), quantile_cont(p, 0.999), max(p)
    FROM (SELECT Patrimonio AS p FROM nav WHERE Patrimonio > 0)
    UNION ALL
    SELECT 'Participes', min(h),
           quantile_cont(h, 0.01), quantile_cont(h, 0.5),
           quantile_cont(h, 0.99), quantile_cont(h, 0.999), max(h)
    FROM (SELECT Participes AS h FROM nav WHERE Participes > 0)
""").df()
)

# Distance to the Silver decimal ceiling (see spark/parsing/fondmens.py)
print("Distance to the Silver decimal ceiling")
duckdb.sql("""
    SELECT max(Patrimonio) AS max_aum,
           1e16 AS decimal_18_2_bound,
           round(100.0 * max(Patrimonio) / 1e16, 6) AS aum_pct_of_bound
    FROM nav
""").df()

Percentiles of the positive range, per measure


,measure,min_positive,p01,p50,p99,p999,max_value
0,VL,0.01,1.0439,1.263380e+01,1.810031e+03,7.673158e+03,1.485200e+04
1,Patrimonio,0.01,3455.0200,1.827739e+07,1.821747e+09,7.582496e+09,1.022021e+10
2,Participes,1.00,1.0000,2.730000e+02,1.046050e+05,3.650729e+05,3.960410e+05


Distance to the Silver decimal ceiling


,max_aum,decimal_18_2_bound,aum_pct_of_bound
0,1.022021e+10,1.000000e+16,0.000102


In [8]:
# Most frequent non-zero values per measure: systemic versus idiosyncratic repeats
for col in ("VL", "Patrimonio", "Participes"):
    print(f"Most frequent non-zero {col} values")
    display(
        duckdb.sql(f"""
        SELECT {col} AS value,
               count(*) AS occurrences,
               count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) AS n_funds
        FROM nav
        WHERE {col} IS NOT NULL AND {col} <> 0
        GROUP BY {col}
        ORDER BY occurrences DESC
        LIMIT 5
    """).df()
    )

Most frequent non-zero VL values


,value,occurrences,n_funds
0,10.0000,2490,121
1,100.0000,1434,89
2,300.0000,1132,21
3,6.0000,1011,54
4,10.9619,540,32


Most frequent non-zero Patrimonio values


,value,occurrences,n_funds
0,300000.00,2291,237
1,0.01,998,13
2,2543.67,508,1
3,-0.01,469,4
4,150000.00,241,49


Most frequent non-zero Participes values


,value,occurrences,n_funds
0,1,155273,978
1,2,92317,583
2,3,54211,504
3,4,40287,465
4,6,34939,432


- **Note:** the ranges are plausible and there is no overflow risk (the maximum AUM uses 0.000102% of the Decimal bound). Three sentinel patterns emerge from the second table:
  - Round NAVs of 10, 100, 300 and 6 euros appear on 21 to 121 funds each. These are most likely funds that distribute income to investors instead of reinvesting it, so the NAV stays anchored at the launch price. A constant NAV is a vehicle design, not an error. The `Participes` table is dominated by low integers (1 to 6 investors), consistent with [§3](#3-what-can-we-check-sign-coherence).
  - The AUM value 2543.67 repeats 508 times on exactly one fund. Most likely a stale or frozen AUM field on a single vehicle, consistent with the frozen-run pattern of [§6](#6-temporal-continuity-and-stale-navs).
  - AUM at exactly -0.01 appears 469 times on 4 funds. Most likely rounding residue written at fund wind-down, and the bulk of [§2](#2-anomaly-flags-raw-counts)'s negative-AUM count.
- **Gate:** nothing new promoted from this section.

## 5. Stability across periods
**Question:** do anomalies concentrate in specific months, or is the mix stable? Same raw flags as [§2](#2-anomaly-flags-raw-counts), split by month.

In [9]:
print("Anomaly flags per period (same raw flags as §2)")
duckdb.sql("""
    SELECT FechaDatos,
           count(*) AS n_rows,
           count(*) FILTER (WHERE VL < 0)         AS negative_vl,
           count(*) FILTER (WHERE Patrimonio < 0) AS negative_aum,
           count(*) FILTER (WHERE VL = 0)         AS zero_vl,
           count(*) FILTER (WHERE Participes = 0) AS zero_holders
    FROM nav
    GROUP BY FechaDatos
    ORDER BY FechaDatos
""").df()

Anomaly flags per period (same raw flags as §2)


,FechaDatos,n_rows,negative_vl,negative_aum,zero_vl,zero_holders
0,202301,91357,0,0,5184,5584
1,202302,82740,0,1,4674,4999
2,202303,92814,0,0,5647,5977
3,202304,90270,0,0,5210,5619
4,202305,92287,0,0,4836,5159
5,202306,89910,0,0,4783,5113
6,202307,93248,0,0,5215,5750
7,202308,93186,0,0,4730,5326
8,202309,90360,0,0,4885,5225
9,202310,93651,0,0,4962,5521


- **Note:** no period is broken. Zero-VL rows hold steady at 4 to 6 thousand per month, around 5% of rows, across the full history. Negative values are confined to a handful of months: negative VL appears in 4 periods (2 rows in 202311, 1 row each in 202503, 202507 and 202604) and negative AUM in 6. Zero-holders shows one clear spike in 202503 (1,133 rows against a typical range of 150 to 600) and a downward trend from roughly 500 rows per month in 2023 to roughly 170 in 2025.
- **Hypothesis:** the stable zero share confirms the padding and dormant-class explanation of [§2](#2-anomaly-flags-raw-counts). The 202503 spike in zero-holders likely reflects a bulk update of the investors field in the source. The overall mix is stable enough to support incremental processing without per-period special-casing.
- **Gate:** nothing new promoted from this section.

## 6. Temporal continuity and stale NAVs
**Question:** does every fund publish every calendar day, and does its NAV actually move?

Context (CNMV PDF): the file has a row for **every day of the month**. Weekend rows still drift slightly day to day, so a run of *identical* consecutive `VL` values is genuinely anomalous. A run breaks when `VL` changes or after a gap of more than 3 days. Zero-VL rows are excluded (already measured in [§2](#2-anomaly-flags-raw-counts)).

In [10]:
# 6a. Calendar coverage: missing days per fund-month (expected: 0)
print("Missing days per fund-month (expected: only 0)")
duckdb.sql("""
    WITH per_month AS (
        SELECT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
               count(DISTINCT value_date) AS days_with_data
        FROM nav
        GROUP BY ALL
    )
    SELECT days_in_month - days_with_data AS missing_days, count(*) AS fund_months
    FROM (
        SELECT *,
               day(last_day(make_date(substr(FechaDatos::VARCHAR, 1, 4)::INT,
                                      substr(FechaDatos::VARCHAR, 5, 2)::INT, 1))) AS days_in_month
        FROM per_month
    )
    GROUP BY 1
    ORDER BY 1
""").df()

Missing days per fund-month (expected: only 0)


,missing_days,fund_months
0,0,125158


- **Note:** coverage is complete: every one of the 125,158 fund-months has a value for every day of the month.
- **Hypothesis:** consistent with the source definition, the file publishes a row per calendar day; no missing-day handling is needed downstream.

In [11]:
# 6b. Frozen-run totals: runs of identical consecutive VL, the numbers promoted to the gate as WARN
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW vl_runs AS
    WITH ordered AS (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase, value_date, VL,
               date_diff('day', lag(value_date) OVER w, value_date) AS gap_days,
               lag(VL) OVER w AS prev_vl
        FROM nav
        WHERE VL IS NOT NULL AND VL > 0
        WINDOW w AS (PARTITION BY Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase ORDER BY value_date)
    ),
    flagged AS (
        SELECT *,
               CASE WHEN gap_days IS NULL OR gap_days > 3 OR VL IS DISTINCT FROM prev_vl THEN 1 ELSE 0 END AS new_run
        FROM ordered
    ),
    runs AS (
        SELECT *,
               sum(new_run) OVER (PARTITION BY Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase ORDER BY value_date) AS run_id
        FROM flagged
    )
    SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase, run_id,
           count(*) AS run_days, min(value_date) AS run_start, max(value_date) AS run_end,
           min(VL) AS vl_min, max(VL) AS vl_max
    FROM runs
    GROUP BY ALL
""")

print("Frozen-run totals: runs and distinct funds by minimum length")
duckdb.sql("""
    SELECT count(*) FILTER (WHERE run_days >= 5)   AS runs_ge5,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE run_days >= 5)   AS funds_ge5,
           count(*) FILTER (WHERE run_days >= 30)  AS runs_ge30,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE run_days >= 30)  AS funds_ge30,
           count(*) FILTER (WHERE run_days >= 120) AS runs_ge120,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) FILTER (WHERE run_days >= 120) AS funds_ge120
    FROM vl_runs
""").df()

Frozen-run totals: runs and distinct funds by minimum length


,runs_ge5,funds_ge5,runs_ge30,funds_ge30,runs_ge120,funds_ge120
0,710,289,98,87,7,6


- **Note:** frozen runs are not rare: 710 runs of 5 or more days across 289 funds, of which 98 last 30 days or more (87 funds) and 7 last 120 days or more (6 funds).
- **Hypothesis:** a frozen run is most likely a suspension during a restructuring, or a vehicle designed with a constant price ([§4](#4-distributions-and-sentinels) already identified the same price pattern among the systemic sentinels). Not an error by itself.
- **Gate:** promoted as WARN.

In [12]:
# 6c. The longest frozen runs, with fund names
print("Longest frozen runs (>= 5 days)")
duckdb.sql("""
    WITH names AS (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
               any_value(Denominacion) AS fund_name
        FROM meta
        GROUP BY 1, 2, 3, 4
    )
    SELECT r.*, n.fund_name
    FROM vl_runs r
    LEFT JOIN names n USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
    WHERE r.run_days >= 5
    ORDER BY r.run_days DESC
    LIMIT 5
""").df()

Longest frozen runs (>= 5 days)


,Tipo,NumeroRegistro,NumeroCompartimento,NumeroClase,run_id,run_days,run_start,run_end,vl_min,vl_max,fund_name
0,FI,5761,0,0,1.0,531,2023-05-05,2024-10-16,10.0000,10.0000,"RENTA VARIABLE INDICE EUROPA, FI"
1,FI,5277,1,0,532.0,322,2025-04-01,2026-02-16,10.9619,10.9619,"MOMENTO, FI"
2,FI,5764,0,0,1.0,271,2023-05-19,2024-02-13,10.0000,10.0000,"RENTA FIJA PRIVADA INDICE EUROPA, FI"
3,FI,4975,4,0,475.0,175,2025-03-26,2025-09-17,2.0745,2.0745,"GESTION BOUTIQUE V, FI"
4,FI,5277,1,0,529.0,160,2024-07-24,2024-12-30,10.8466,10.8466,"MOMENTO, FI"


- **Note:** the longest freezes sit at exactly 10.0000 euros for over a year, and affect recognisable vehicles (index funds, boutique strategies).
- **Hypothesis:** confirms the constant-round-NAV reading of [§4](#4-distributions-and-sentinels); each fund should be confirmed before excluding it from return calculations.

In [13]:
# 6d. Day-over-day NAV change: tail profile and the largest jumps
print("Day-over-day |change| profile")
display(
    duckdb.sql("""
    WITH d AS (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase, ISIN, value_date, VL,
               VL / NULLIF(lag(VL) OVER w, 0) - 1 AS daily_change
        FROM nav
        WHERE VL > 0
        WINDOW w AS (PARTITION BY Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase ORDER BY value_date)
    )
    SELECT count(*) AS observations,
           quantile_cont(abs(daily_change), 0.5)   AS abs_p50,
           quantile_cont(abs(daily_change), 0.99)  AS abs_p99,
           quantile_cont(abs(daily_change), 0.999) AS abs_p999,
           max(abs(daily_change))                  AS abs_max
    FROM d
    WHERE daily_change IS NOT NULL
""").df()
)

print("Three largest day-over-day jumps")
duckdb.sql("""
    WITH d AS (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase, ISIN, value_date, VL,
               VL / NULLIF(lag(VL) OVER w, 0) - 1 AS daily_change
        FROM nav
        WHERE VL > 0
        WINDOW w AS (PARTITION BY Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase ORDER BY value_date)
    )
    SELECT * FROM d
    WHERE daily_change IS NOT NULL
    ORDER BY abs(daily_change) DESC
    LIMIT 3
""").df()

Day-over-day |change| profile


,observations,abs_p50,abs_p99,abs_p999,abs_max
0,3600037,0.000468,0.018659,0.040241,269.35105


Three largest day-over-day jumps


,Tipo,NumeroRegistro,NumeroCompartimento,NumeroClase,ISIN,value_date,VL,daily_change
0,FI,5619,3,0,ES0133627028,2023-11-13,2549.3293,269.351050
1,FI,5138,2,0,ES0131444038,2024-02-28,14852.0000,141.807692
2,FI,5619,1,0,ES0133627002,2024-08-09,7.2437,7.108014


- **Note:** day-to-day changes are small for the vast majority of rows (median 0.05%, p99 under 2%), with a thin tail of extreme jumps concentrated on two funds.
- **Hypothesis:** the largest jump (+26,935% on FI-5619) comes the day after that same fund's two negative-VL rows, so it is best read as a restatement or recovery glitch rather than a real performance event. The second jump produces the dataset's maximum NAV, which suggests a unit change or share consolidation. Both are idiosyncratic.
- **Gate:** deferred to the anomaly-detection objective.

## 7. Referential integrity and ISIN quality
**Question:** do the two Silver tables agree, is the ISIN stable per class, and are ISINs well-formed?

In [14]:
# 7a. Both directions of the NAV <-> metadata join, class-month level
nav_keys = duckdb.sql("""
    SELECT DISTINCT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase FROM nav
""").df()
meta_keys = duckdb.sql("""
    SELECT DISTINCT FechaDatos, Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase FROM meta
""").df()

import pandas as pd

nav_km = pd.MultiIndex.from_frame(nav_keys)
meta_km = pd.MultiIndex.from_frame(meta_keys)
only_nav = nav_km.difference(meta_km)
only_meta = meta_km.difference(nav_km)
print(f"nav class-months: {len(nav_km):,}; meta class-months: {len(meta_km):,}")
print(f"nav without meta: {len(only_nav)} (gate FAIL, expected 0)")
print(
    f"meta without nav: {len(only_meta)} class-months, "
    f"{len({k[1:] for k in only_meta}):,} funds"
)

nav class-months: 125,158; meta class-months: 125,807
nav without meta: 0 (gate FAIL, expected 0)
meta without nav: 649 class-months, 404 funds


- **Note:** the NAV-to-metadata direction is clean, no class-month without a metadata row, which confirms the gate check. The reverse direction is not: 649 metadata class-months, covering 404 funds, have no NAV rows at all, around 0.5% of the metadata.
- **Hypothesis:** those are registered funds that never published a daily series in the period, expected in a registry that includes not-yet-trading vehicles rather than a data loss.
- **Gate:** promoted as WARN.

In [15]:
# 7b. ISIN invariance: one class key with several ISINs, or one ISIN on several keys
print("Class keys that changed ISIN during the history")
display(
    duckdb.sql("""
    SELECT count(*) AS keys_with_multiple_isins
    FROM (
        SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase
        FROM nav
        WHERE ISIN IS NOT NULL AND trim(ISIN) <> ''
        GROUP BY ALL
        HAVING count(DISTINCT ISIN) > 1
    )
""").df()
)
print("ISINs that appear on more than one class key")
display(
    duckdb.sql("""
    SELECT count(*) AS isins_on_multiple_keys
    FROM (
        SELECT ISIN
        FROM nav
        WHERE ISIN IS NOT NULL AND trim(ISIN) <> ''
        GROUP BY ISIN
        HAVING count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)) > 1
    )
""").df()
)

# The keys that changed ISIN during the history, with the transition date
print("Example: keys that changed ISIN (with dates)")
duckdb.sql("""
    SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
           count(DISTINCT ISIN) AS n_isins,
           string_agg(DISTINCT ISIN, ', ' ORDER BY ISIN) AS isins,
           min(value_date) AS first_seen, max(value_date) AS last_seen
    FROM nav
    WHERE ISIN IS NOT NULL AND trim(ISIN) <> ''
    GROUP BY ALL
    HAVING count(DISTINCT ISIN) > 1
    ORDER BY first_seen
    LIMIT 5
""").df()

Class keys that changed ISIN during the history


,keys_with_multiple_isins
0,6


ISINs that appear on more than one class key


,isins_on_multiple_keys
0,93


Example: keys that changed ISIN (with dates)


,Tipo,NumeroRegistro,NumeroCompartimento,NumeroClase,n_isins,isins,first_seen,last_seen
0,FI,5774,0,2,2,"ES0162370011, ES0162370029",2025-01-01,2026-05-31
1,FI,5771,0,2,2,"ES0162369013, ES0162369021",2025-01-01,2026-05-31
2,FI,5774,0,1,2,"ES0162370003, ES0162370011",2025-01-01,2026-05-31
3,FI,5771,0,1,2,"ES0162369005, ES0162369013",2025-01-01,2026-05-31
4,FI,5316,4,1,2,"ES0131445159, ES0131445167",2025-03-01,2026-05-31


- **Note:** a handful of class keys changed ISIN during the history (the example shows the transition and its effective date), and a larger group of ISINs appears on more than one class key.
- **Hypothesis:** an ISIN change is most likely a re-registration or reassignment, and must be understood before joining history on ISIN. Multiple keys sharing one ISIN is legitimate when the source XSD defines the ISIN at compartimento or fund level rather than at class level, so this is a property of the source rather than a defect.
- **Gate:** both patterns were promoted as WARN.

In [16]:
# 7c. ISIN quality in metadata: null / empty / bad format, same regex the gate uses
print("ISIN quality counts")
display(
    duckdb.sql("""
    SELECT count(*) AS total_meta_rows,
           count(*) FILTER (WHERE ISIN IS NULL)   AS isin_null,
           count(*) FILTER (WHERE trim(ISIN) = '') AS isin_empty,
           count(*) FILTER (WHERE ISIN IS NOT NULL AND trim(ISIN) <> ''
                            AND NOT regexp_matches(trim(ISIN), '^[A-Z]{2}[A-Z0-9]{9}[0-9]$')) AS isin_bad_format
    FROM meta
""").df()
)

print("Rows with an empty ISIN")
duckdb.sql("""
    SELECT DISTINCT ISIN, Denominacion, FechaDatos
    FROM meta
    WHERE ISIN IS NOT NULL AND trim(ISIN) = ''
    ORDER BY FechaDatos
""").df()

ISIN quality counts


,total_meta_rows,isin_null,isin_empty,isin_bad_format
0,125807,0,2,0


Rows with an empty ISIN


,ISIN,Denominacion,FechaDatos
0,,"GVCGAESCO JAPON, FI",202403
1,,"LABORAL KUTXA AHORRO,FI",202403


- **Note:** metadata ISINs are otherwise clean, no nulls, no bad lengths, no format violations: only 2 rows carry an empty ISIN, both in the same period.
- **Hypothesis:** a source omission on a single reporting batch rather than a systemic defect; already covered by the gate's format WARN.

## 8. Findings and decisions (conclusion)

This table records what was found, at what scale, what to do about it and where each rule is enforced. `Kind` values: **structural** = schema, key, null and referential-integrity checks, the only tier allowed to FAIL; **semantic** = financial anomalies, never blocking — their filtering and quarantine live in dbt. `Destination` values: **gate FAIL** blocks the pipeline (reachable only from structural); **gate WARN** reports and continues; **quarantine** holds rows out of the analytical layer (deferred to dbt); **documented-only** is reported for observability and never affects the exit code.

Negative values are split by a material/dust threshold (`NEG_DUST_MAX = 1`): `abs(v) > 1` is a material anomaly, a row to quarantine in dbt; `0 < abs(v) <= 1` is rounding dust. Both are reported as gate WARN; neither blocks the gate.

| Anomaly | Rows (funds) | Action | Kind | Destination |
|---|---|---|---|---|
| Nulls in any column | 0 | — | semantic | documented-only |
| Duplicate keys | 0 | — | structural | gate FAIL (passing) |
| Null keys / null mandatory fields (nav + meta) | 0 | — | structural | gate FAIL (passing) |
| value_date month vs FechaDatos | 0 | — | structural | gate FAIL (passing) |
| Silver schema drift (nav + metadata) | 0 | block any column/type change vs the parsers' EXPECTED_SCHEMA | structural | gate FAIL (passing) |
| NAV class-month without metadata | 0 | — | structural | gate FAIL (passing) |
| Negative VL, material | 5 (5) | quarantine the glitch days (FI-5619) in dbt | semantic | gate WARN |
| Negative VL, dust | 0 | — | semantic | gate WARN (passing) |
| Negative AUM, material | 43 (7) | quarantine with the dbt layer | semantic | gate WARN |
| Negative AUM, dust | 479 (7) | mostly the -0.01 residue (469 rows, 4 funds) | semantic | gate WARN |
| Negative holders, material | 0 | — | semantic | gate WARN (passing) |
| Negative holders, dust | 10 (1) | all -1 rows in December 2023 | semantic | gate WARN |
| Zero VL | 202,963 (1,089) | flag zeros downstream | semantic | gate WARN |
| Sign contradictions (3 combinations) | 2,080 + 15,585 + 11 | flag rows; recurring source pattern | semantic | gate WARN |
| Frozen VL runs of 5 or more days | 710 runs (289) | monitor; confirm per fund before excluding | semantic | gate WARN |
| ISIN changed within key | 6 keys | check before joining history on ISIN | semantic | gate WARN |
| Metadata class-month without NAV | 649 (404) | accept; monitor freshness | semantic | gate WARN |
| ISIN reused across keys | 93 ISINs | accept (allowed by source XSD) | semantic | gate WARN |
| Empty ISIN in metadata | 2 (2) | source omission in period 202403; covered by the format WARN | semantic | gate WARN |
| Negative rows (material + dust) | 537 | quarantine deferred to dbt; SKIP in the gate | semantic | quarantine (deferred) |

**Source of truth:** the `FINDINGS` contract in `spark/jobs/validate_silver.py`. Every gate check is one entry there (id, kind, destination, level, check function); this table is the readable mirror.

**Gate state:** 0 FAIL, 13 WARN, 11 PASS, 1 INFO, 1 SKIP, exit 0 (26 findings): structural 0 FAIL / 9 PASS, semantic 13 WARN / 2 PASS. The former FAILs — material negatives (5 VL + 43 AUM rows) — are WARN now: the gate no longer blocks on anomalies whose filtering and quarantine belong to the dbt layer.

**Pre-publication checklist**

- [ ] Restart and Run All works top to bottom on a fresh kernel
- [ ] findings table matches the executed outputs
- [ ] notebook committed with outputs (policy: exploration notebooks always keep outputs so they render on GitHub)